To do : 

implement R-score for default ps.SINDy parameters
- find out how to vary ps.SINDy threshold values
- find out how ps.SINDy implements SLTSQ
Notes :
- if you want to implement an integrator yourself, you need to define a function that is parametrised by the selected coefficients and that returns a combination of all the library functions for each component.

Good to know : 
ps.SINDy location 
C:\\Users\\abdel\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\pysindy\\_core.py

Note : SINDy imports STLSQ from .optimizers

Essence of model.fit:
class SINDy(_BaseSINDy): 
        ....
        def fit(...):
        ...
        
        self.feature_names = feature_names

        x_dot = concat_sample_axis(x_dot)
        x = self.feature_library.fit_transform(x)
        x = SampleConcatter().fit_transform(x)
        self.optimizer.fit(x, x_dot)
        self._fit_shape()

        return self

Im so confused : cimplicated code in stlsq.py , where's the actual optimization 'happening'/supposed to happen?

In [154]:
from noising_lorenz_data import *

In [164]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from pysindy import SINDy
from pysindy.optimizers import STLSQ
import inspect

In [ ]:
dataframe = pd.read_csv("./lorenz_data.csv")
data = dataframe.to_numpy()
data_shape = data.shape

In [186]:
gnoise = np.random.normal(0,100,data_shape)

t = data[:,0]
x = data[:,1:4]
xdot = data[:,4:]
xnoise = x + gnoise[:,1:4]
xdotnoise = xdot + gnoise[:,4:]

Maybe show the coefficients of the actual model for each sparsity threshold

In [187]:
n_range=25
initial_point = [0,0,1]
opts = [STLSQ(threshold=10**(-i), alpha=.5) for i in range(1,n_range)]
models = [SINDy(optimizer=opt) for opt in opts] # generally we can "sweep" over different parameters randomly, see SINDy4Engineers
# default feature names are polynomials, see SINDy class core.py
Rsqs = []
for i in range(1,n_range):
    model = models[i-1]
    model.fit(x,t,xdot,feature_names=['x','y','z'])
    coeffs0 = model.coefficients()
    sample_path0 = model.simulate(initial_point,t[9900:-1])
    
    model.fit(xnoise,t,feature_names=['x','y','z'])
    coeffsnoise1 = model.coefficients()
    sample_path1 = model.simulate(initial_point,t[9900:-1])
    
    SSE = np.sum((sample_path0[0]-sample_path1[0])**2)
    avg_groundtruth = np.sum(sample_path0[0])/len(sample_path0)
    SST = np.sum((sample_path0[0]-avg_groundtruth)**2)
    Rsq = 1-(SSE/SST)
    print(Rsq)
    Rsqs.append(Rsq)

plt.plot([i for i in range(1,n_range)], Rsqs)
plt.show()

1.0
1.0
1.0
1.0
1.0


KeyboardInterrupt: 

First succesful run : i from 1 to 10 gives an output of all 1.0 for Rsq. looks like R-squared is a poor measure or something is wrong with the code. All models have "perfect" fit, which is unlikely.

Second run, now with i from 1 to 25, the output for Rsq is only 1's

Maybe the added noise is insignificant?

We need to compare the actual model with the predicted model(s)
For each model, we have the information we need from the coefficients and the feature library. 
We want to numerically integrate these models to see how they compare to each other.
Choose some point x,y,z and take it as starting value.